In [1]:
import os
from promptsmith.dspy_init import get_dspy
import dspy
import textgrad as tg

In [2]:
from promptsmith.tasks.bad_to_good.bulletize_text_2 import BulletizeText
from promptsmith.judges.judge_bullet_structure import JudgeBulletStructure
from promptsmith.judges.judge_coverage import JudgeCoverage
from promptsmith.judges.judge_focus_relevance import JudgeFocusRelevance
from promptsmith.judges.judge_redundancy import JudgeRedundancy
from promptsmith.refiners.feedback_aggregator import FeedbackAggregator

In [3]:
dspy, lm = get_dspy()

In [4]:
tg.set_backward_engine("gpt-4o-mini", override=True)

In [5]:
bulletizer = dspy.ChainOfThought(BulletizeText)
judges = {
    "structure": dspy.Predict(JudgeBulletStructure),
    "coverage": dspy.Predict(JudgeCoverage),
    "focus_relevance": dspy.Predict(JudgeFocusRelevance),
    "redundancy": dspy.Predict(JudgeRedundancy),
}
weights = {"structure": 0.5, "coverage": 0.2, "focus_relevance": 0.15, "redundancy": 0.15}

In [6]:
def evaluate_with_judges(input_text: str, output_text: str):
    print("\n=== Evaluating with Judges ===")
    print("Input text:\n", input_text[:120].replace("\n", " "), "..." if len(input_text) > 120 else "")
    print("Output text:\n", output_text[:120].replace("\n", " "), "..." if len(output_text) > 120 else "")
    scores = {}
    reasonings = {}
    for name, judge in judges.items():
        print(f"\nRunning judge: {name}")
        if "input_text" in judge.signature.fields:
            res = judge(input_text=input_text, output_text=output_text)
        else:
            res = judge(output_text=output_text)
        scores[name] = float(res.score)
        reasonings[name] = str(res.reasoning)
        print(f"  Score: {scores[name]}")
        print(f"  Reasoning:\n{reasonings[name]}")
    combined = sum(weights[k] * scores.get(k, 0.0) for k in weights)
    print(f"\nCombined weighted score: {combined:.3f}")
    feedback_lines = []
    for k in ["structure", "coverage", "focus_relevance", "redundancy"]:
        if k in reasonings:
            feedback_lines.append(f"- {k}: {reasonings[k]}")
    feedback = "\n".join(feedback_lines)

    feedback_aggregator = dspy.Predict(FeedbackAggregator)
    combined_feedback_aggregated = feedback_aggregator(
        original_text=input_text,
        current_output=output_text,
        task_type='bulletize',
        judges_feedback=[reasonings[k] for k in ["structure", "coverage", "focus_relevance", "redundancy"] if k in reasonings]
    ).combined_feedback

    print("\nFeedback summary:")
    # print(feedback)
    print(combined_feedback_aggregated)
    print("=== End Evaluation ===\n")
    return combined, combined_feedback_aggregated, scores, reasonings

In [32]:
with open('../../data/text_01.txt', 'r') as f:
    text = f.read()

In [52]:
text = """
Patients with a chronic disease may experience problems with mobility, fatigue, pain, elimination, social isolation, loneliness, depression, and more, which can negatively affect quality of life. There’s a plethora of information in the literature on chronic disease management and how people cope with chronic illness. The following are examples of selected models and processes.
2. False normality—The person achieves a partial life balance and control, but this may not be sustainable over time. 3. New normal—The person has embraced a positive approach to coping with his or her disease and lives a normal life. 4. Disruptive—The person experiences a disruption, which can be physical, mental, social, family-oriented, or financial.
1. Acceptance—The person is knowledgeable about his or her disease and is willing to cope with the uncertainty that the diagnosis brings. 2. Coping—The person comes to terms with the reality of the chronic disease and develops new ways to adjust to it. 3. Self-management—The person is proactive in his or her care.
2. Delivery system—Provide efficient clinical care to keep patients out of crisis, including encouraging self-care by supplying available resources. 3. Self-management support—Empower the patient to take an active role in his or her care. 4. Decision-support—Make healthcare decisions based on evidence-based standards of practice, with support from the interprofessional team. 5. Clinical information systems—Properly and securely store, retrieve, and analyze patient-centered data to provide efficient and timely care.
"""

In [53]:
print(text)


Patients with a chronic disease may experience problems with mobility, fatigue, pain, elimination, social isolation, loneliness, depression, and more, which can negatively affect quality of life. There’s a plethora of information in the literature on chronic disease management and how people cope with chronic illness. The following are examples of selected models and processes.
2. False normality—The person achieves a partial life balance and control, but this may not be sustainable over time. 3. New normal—The person has embraced a positive approach to coping with his or her disease and lives a normal life. 4. Disruptive—The person experiences a disruption, which can be physical, mental, social, family-oriented, or financial.
1. Acceptance—The person is knowledgeable about his or her disease and is willing to cope with the uncertainty that the diagnosis brings. 2. Coping—The person comes to terms with the reality of the chronic disease and develops new ways to adjust to it. 3. Self-m

In [35]:
bulletized_text = bulletizer(input_text=text).output_text
bulletized_text

"# Faculty Perceptions of Online Teaching Competencies\n\nThis study explores the relationship between faculty perceptions of online teaching competencies and their ability to teach online.\n\n## Research Question 1: Relationship Between Perceptions and Ability\n**What is the strength and direction of the relationship between faculty's perceptions of competencies and their ability to teach online?**\n- Descriptive statistics (mean and standard deviation) reported in Table 2 on page 23.\n- High ratings for both constructs, Importance and Ability, across competencies.\n  - **Course Design:**\n    - Manage grades online (M = 4.73)\n    - Creating online assignments (M = 4.68)\n  - **Course Communication:**\n    - Responding to student questions promptly (M = 4.79)\n    - Providing feedback on assignments (M = 4.65)\n  - **Time Management:**\n    - Scheduling time to design the course prior to delivery (M = 4.65)\n    - Spending weekly hours to grade assignments (M = 4.54)\n  - **Technical

In [44]:
combined, feedback, scores, reasonings = evaluate_with_judges(text, bulletized_text)

print("Outputs:")
print("\nCombined Score:\n", combined)
print("\nFeedback:\n", feedback)
print("\nScores:\n", scores)
print("\nReasonings:\n", reasonings)


=== Evaluating with Judges ===
Input text:
 The first question in this study stated: What is the strength and direction of the relationship between faculty's percep ...
Output text:
 # Faculty Perceptions of Online Teaching Competencies  This study explores the relationship between faculty perceptions  ...

Running judge: structure
  Score: 0.65
  Reasoning:
- The title is appropriate and follows the guidelines.
- The one-line summary is present and meets the requirements.
- Section heads are correctly formatted with meaningful H2 headings.
- The bullet points are mostly concise and start with appropriate symbols, but there are some issues:
  - The first section has too many top-level bullets (more than 8).
  - The nested sub-bullets under "Course Design," "Course Communication," "Time Management," and "Technical Competency" are not consistently formatted; some use bold labels while others do not.
- Each section has a closing line, but the closing lines do not follow the required form

In [7]:
def refine_prompt_with_textgrad(input_text: str, init_prompt: str = "", steps: int = 5, data_collector=None, example_info=None):
    print("\n[TextGrad] Starting prompt refinement process...")
    
    start_time = time.time()
    all_steps = []
    max_score = 0.0
    best_step = 0

    # Editable variable: the prompt/instruction injected into the task docstring
    prompt = tg.Variable(init_prompt or "You are an expert technical writer.", role_description="Instruction that guides the bulletizer to follow the rules precisely")

    def make_bulletizer(prompt_text: str):
        base_sig = BulletizeText
        full_doc = (prompt_text.strip() + "\n\n" + (base_sig.__doc__ or "")).strip()
        DynamicSig = type(
            "BulletizeTextPrompt",
            (dspy.Signature,),
            {
                "__doc__": full_doc,
                "input_text": base_sig.model_fields["input_text"],
                "output_text": base_sig.model_fields["output_text"],
            },
        )
        return dspy.Predict(DynamicSig)

    optim = tg.TGD(parameters=[prompt])

    for i in range(steps):
        step_start_time = time.time()
        print(f"\n[TextGrad] === Step {i+1}/{steps} ===")
        bulletizer_pred = make_bulletizer(prompt.value)

        print("[TextGrad] Running bulletizer with current prompt...")
        current_output = bulletizer_pred(input_text=input_text).output_text
        print("[TextGrad] Current output:\n", current_output, "\n")

        print("[TextGrad] Evaluating output with judges...")
        eval_start_time = time.time()
        combined, feedback, scores, reasonings = evaluate_with_judges(input_text, current_output)
        eval_time = time.time() - eval_start_time
        print(f"[TextGrad] Combined judge score: {combined:.3f}")
        print("[TextGrad] Judge feedback to address:\n", feedback, "\n")

        # Track best score
        if combined > max_score:
            max_score = combined
            best_step = i + 1

        print("[TextGrad] Computing textual gradient to improve the PROMPT...")
        textgrad_start_time = time.time()
        evaluation_instruction = (
            "You are revising the INSTRUCTION/PROMPT that guides a bulletization task.\n"
            "Goal: Modify the prompt so that, when used to bulletize the same input text, the output fully satisfies the rules and addresses judges' feedback.\n\n"
            f"Input text:\n{input_text}\n\n"
            f"Current prompt/instruction:\n{prompt.value}\n\n"
            f"Output produced with this prompt:\n{current_output}\n\n"
            "Judges' feedback (issues to fix via better prompt wording/emphasis):\n"
            f"{feedback}\n\n"
            "Revise ONLY the prompt text. Be explicit, unambiguous, and operational about structure, coverage, focus, and redundancy. "
            "Output only the revised prompt."
        )
        loss = tg.TextLoss(evaluation_instruction)(prompt)
        loss.backward()
        optim.step()
        textgrad_time = time.time() - textgrad_start_time

        step_total_time = time.time() - step_start_time

        # Create step data for CSV
        if data_collector and example_info:
            step_data = TextGradStep(
                step_number=i + 1,
                structure_score=scores.get("structure", 0.0),
                coverage_score=scores.get("coverage", 0.0),
                focus_relevance_score=scores.get("focus_relevance", 0.0),
                redundancy_score=scores.get("redundancy", 0.0),
                combined_score=combined,
                structure_reasoning=reasonings.get("structure", ""),
                coverage_reasoning=reasonings.get("coverage", ""),
                focus_relevance_reasoning=reasonings.get("focus_relevance", ""),
                redundancy_reasoning=reasonings.get("redundancy", ""),
                combined_feedback=feedback,
                output=current_output,
                refinement_reasoning="N/A",  # TextGrad doesn't use refiner
                is_final_step=(i == steps - 1),
                converged_early=False,  # TextGrad doesn't converge early
                evaluation_time=eval_time,
                aggregation_time=0.0,  # TextGrad doesn't aggregate feedback
                refinement_time=textgrad_time,
                total_step_time=step_total_time
            )
            all_steps.append(step_data)

        print("[TextGrad] Updated prompt:\n", prompt.value, "\n")

    print("[TextGrad] Final run with improved prompt...")
    final_bulletizer = make_bulletizer(prompt.value)
    final_output = final_bulletizer(input_text=input_text).output_text

    print("[TextGrad] Final evaluation with judges...")
    final_score, _, final_scores, final_reasons = evaluate_with_judges(input_text, final_output)
    print("[TextGrad] Final combined score:", final_score)

    total_time = time.time() - start_time

    # Create final result for data collector
    if data_collector and example_info:
        result = TextGradResult(
            example_id=example_info.get("example_id", "unknown"),
            original_id=example_info.get("original_id", "unknown"),
            title=example_info.get("title", ""),
            profession=example_info.get("profession", ""),
            purpose=example_info.get("purpose", ""),
            original_text_length=len(input_text),
            steps=all_steps,
            total_steps=steps,
            max_combined_score=max_score,
            step_of_max_combined_score=best_step,
            total_textgrad_time=total_time,
            avg_time_per_step=total_time / steps
        )
        data_collector.add_result(result)

    return {
        "improved_prompt": prompt.value,
        "final_outline": final_output,
        "final_score": final_score,
        "per_judge_scores": final_scores,
        "per_judge_reasonings": final_reasons,
    }

In [10]:
bulletize_text_prompt = """
    You are an expert technical writer.

    **Task**  
    Rewrite the supplied text as a structured, Markdown-formatted outline.

    **Title rule (T)**  
    • If the source already has a clear title **or** the text spans several distinct sections (e.g., report, article, interview), add one H1 heading (`# …`) at the very top.  
    • Keep the title ≤ 8 words and descriptive.  
    • Otherwise, skip the title entirely.

    **Format rules**  
    0. Start with one sentence (≤ 15 words) that summarizes the entire text. *Do not add any label.*  
    1. Add an H2 heading (`## …`) for each logical block.  
    2. Under every heading supply **3-8** concise bullets:  
        • capture every unique name, date, statistic, money figure, or quote (< 20 words)  
        • if the source pairs values (e.g., Importance vs Ability), show both in one bullet  
        • begin a bullet with a **bold label** when the sentence naturally has one (e.g., **Time Management:**)  
        • nest bullets where helpful to show hierarchy or examples  
        • if a block would have < 3 bullets, merge it with a neighbor or expand a point so the section stands alone  
    3. When the source contains an explicit question, place that question on its own line in **bold** right before the bullets that answer it.  
    4. Strip filler, greetings, ads, and repetition.  
    5. For transcripts, group by topic (not speaker turns) and omit sponsor segments.  
    6. End each section with one italic sentence that summarizes the block. *Do not write “Takeaway:”.*  
    7. Separate major sections with three dashes (`---`) on a line by themselves.  
    8. Output only the formatted outline—no extra commentary.
"""

In [55]:
result = refine_prompt_with_textgrad(text, bulletize_text_prompt)
print(result)


[TextGrad] Starting prompt refinement process...

[TextGrad] === Step 1/5 ===
[TextGrad] Running bulletizer with current prompt...
[TextGrad] Current output:
 # Chronic Disease Management and Coping Models

Patients with chronic diseases face various challenges that impact their quality of life.

## Challenges Faced by Patients
- **Common Issues:** Mobility, fatigue, pain, elimination problems, social isolation, loneliness, depression.
- **Impact:** These challenges can negatively affect overall quality of life.

*Chronic diseases lead to multiple challenges that require effective management strategies.*

---

## Coping Models for Chronic Illness
- ** ...

[TextGrad] Evaluating output with judges...

=== Evaluating with Judges ===
Input text:
  Patients with a chronic disease may experience problems with mobility, fatigue, pain, elimination, social isolation, lo ...
Output text:
 # Chronic Disease Management and Coping Models  Patients with chronic diseases face various challenges tha

In [11]:
# TextGrad Data Collection Classes (matching refinement script format)
from dataclasses import dataclass
from typing import List, Dict, Any
from pathlib import Path
import csv
import json
import time

@dataclass
class TextGradStep:
    """Represents a single step in the TextGrad refinement process."""
    
    step_number: int
    structure_score: float
    coverage_score: float
    focus_relevance_score: float
    redundancy_score: float
    combined_score: float
    structure_reasoning: str
    coverage_reasoning: str
    focus_relevance_reasoning: str
    redundancy_reasoning: str
    combined_feedback: str  # Aggregated feedback from all judges
    output: str
    refinement_reasoning: str  # Will be "N/A" for TextGrad
    is_final_step: bool
    converged_early: bool  # Will be False for TextGrad
    
    # Timing fields
    evaluation_time: float  # Time for all judges to evaluate
    aggregation_time: float  # Will be 0.0 for TextGrad (no aggregation)
    refinement_time: float  # Time for TextGrad step
    total_step_time: float  # Total time for this step

@dataclass
class TextGradResult:
    """Represents the complete TextGrad process for a single example."""
    example_id: str
    original_id: str
    title: str
    profession: str
    purpose: str
    original_text_length: int
    steps: List[TextGradStep]
    total_steps: int
    max_combined_score: float
    step_of_max_combined_score: int
    
    # Timing fields
    total_textgrad_time: float  # Total time for entire TextGrad process
    avg_time_per_step: float  # Average time per step

class TextGradDataCollector:
    """Collects and manages TextGrad data for multiple examples."""
    
    def __init__(self):
        self.results: List[TextGradResult] = []
        self.example_best_scores: Dict[str, float] = {}
    
    def add_result(self, result: TextGradResult):
        """Add a TextGrad result to the collection."""
        self.results.append(result)
    
    def get_all_steps_for_csv(self) -> List[Dict[str, Any]]:
        """Convert all results to a flat list suitable for CSV export in long format."""
        csv_rows = []
        
        for result in self.results:
            current_best = 0.0
            current_best_step = 0
            
            for step in result.steps:
                # Update current best for this example
                if step.combined_score > current_best:
                    current_best = step.combined_score
                    current_best_step = step.step_number
                
                row = {
                    'example_id': result.example_id,
                    'original_id': result.original_id,
                    'title': result.title,
                    'profession': result.profession,
                    'purpose': result.purpose,
                    'original_text_length': result.original_text_length,
                    'iteration_number': step.step_number,  # Map step to iteration for compatibility
                    'structure_score': round(step.structure_score, 4),
                    'coverage_score': round(step.coverage_score, 4),
                    'focus_relevance_score': round(step.focus_relevance_score, 4),
                    'redundancy_score': round(step.redundancy_score, 4),
                    'combined_score': round(step.combined_score, 4),
                    'structure_reasoning': step.structure_reasoning,
                    'coverage_reasoning': step.coverage_reasoning,
                    'focus_relevance_reasoning': step.focus_relevance_reasoning,
                    'redundancy_reasoning': step.redundancy_reasoning,
                    'combined_feedback': step.combined_feedback,
                    'output': step.output,
                    'refinement_reasoning': step.refinement_reasoning,  # "N/A" for TextGrad
                    'is_final_iteration': step.is_final_step,
                    'converged_early': step.converged_early,  # False for TextGrad
                    'total_iterations': result.total_steps,
                    'current_best_score': round(current_best, 4),
                    'current_best_iteration': current_best_step,
                    # Timing fields
                    'evaluation_time': round(step.evaluation_time, 3),
                    'aggregation_time': round(step.aggregation_time, 3),  # 0.0 for TextGrad
                    'refinement_time': round(step.refinement_time, 3),
                    'total_iteration_time': round(step.total_step_time, 3),
                    'total_refinement_time': round(result.total_textgrad_time, 3),
                    'avg_time_per_iteration': round(result.avg_time_per_step, 3)
                }
                csv_rows.append(row)
        
        return csv_rows
    
    def save_to_csv(self, output_path: Path):
        """Save all results to CSV in long format."""
        csv_rows = self.get_all_steps_for_csv()
        
        if not csv_rows:
            print("No data to save")
            return
        
        fieldnames = csv_rows[0].keys()
        
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(csv_rows)
        
        print(f"Saved {len(csv_rows)} rows to {output_path}")
    
    def get_summary_stats(self) -> Dict[str, Any]:
        """Get summary statistics for all TextGrad results."""
        if not self.results:
            return {}
        
        total_examples = len(self.results)
        total_steps = sum(r.total_steps for r in self.results)
        avg_steps = total_steps / total_examples
        avg_final_score = sum(r.max_combined_score for r in self.results) / total_examples
        
        # Timing statistics
        total_time = sum(r.total_textgrad_time for r in self.results)
        avg_time_per_example = total_time / total_examples
        avg_time_per_step = sum(r.avg_time_per_step for r in self.results) / total_examples
        
        return {
            'total_examples': total_examples,
            'total_iterations': total_steps,
            'average_iterations_per_example': avg_steps,
            'average_final_score': avg_final_score,
            'examples_converged_early': 0,  # TextGrad doesn't converge early
            'convergence_rate': 0.0,  # TextGrad doesn't converge early
            # Timing statistics
            'total_time': total_time,
            'average_time_per_example': avg_time_per_example,
            'average_time_per_iteration': avg_time_per_step,
        }


In [12]:
import os
import json
import time
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# Config
csv_path = Path("/Users/yanivgal/dev/ai21/promptsmith/data/documents_train.csv")
n_samples = 50
text_col = "resource"  # input text column in the CSV
steps_per_example = 5  # TextGrad refinement steps
random_seed = 42

# Output - using same naming convention as refinement script
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = Path(f"/Users/yanivgal/dev/ai21/promptsmith/r_{n_samples}_{steps_per_example}_textgrad_{stamp}.csv")

print(f"Loading: {csv_path}")
df = pd.read_csv(csv_path)

if text_col not in df.columns:
    raise ValueError(f"Expected column '{text_col}' not found. Columns: {list(df.columns)}")

# Sample rows
sampled = df.sample(n=n_samples, random_state=random_seed).reset_index(drop=True)

# Initialize data collector (same format as refinement script)
data_collector = TextGradDataCollector()

start = time.time()
for i, row in tqdm(sampled.iterrows(), total=len(sampled), desc="TextGrad Refining"):
    # Extract fields (best-effort, some columns may be missing)
    original_id = row.get("id") or row.get("resource_id") or row.get("Unnamed: 0") or f"row_{i}"
    title = row.get("title") if "title" in row else ""
    profession = row.get("user_profession") or row.get("user_kyc_profession") or ""
    purpose = row.get("user_write_purpose") or row.get("user_write_purpose (work, school, personal)") or ""

    input_text = str(row[text_col]).strip()
    if not input_text:
        continue

    # Prepare example info for data collection
    example_info = {
        "example_id": f"example_{i+1:03d}",
        "original_id": original_id,
        "title": title,
        "profession": profession,
        "purpose": purpose,
    }

    # Run TextGrad prompt refinement for this sample with data collection
    try:
        res = refine_prompt_with_textgrad(
            input_text=input_text,
            init_prompt="You are an expert technical writer.",
            steps=steps_per_example,
            data_collector=data_collector,
            example_info=example_info
        )
        print(f"\n✅ Completed example {i+1}: Final score = {res['final_score']:.3f}")
        
    except Exception as e:
        print(f"\n❌ Error processing example {i+1}: {e}")

elapsed = time.time() - start

# Save CSV in same format as refinement script
print(f"\nSaving detailed results to {out_csv}...")
data_collector.save_to_csv(out_csv)

print("\n" + "=" * 80)
print("TEXTGRAD PROCESS COMPLETE")
print("=" * 80)

# Get summary stats (same format as refinement script)
summary_stats = data_collector.get_summary_stats()
if summary_stats:
    print(f"Successfully processed: {summary_stats['total_examples']} examples")
    print(f"Average final score: {summary_stats['average_final_score']:.3f}")
    print(f"Average steps per example: {summary_stats['average_iterations_per_example']:.1f}")
    print(f"Total time: {summary_stats['total_time']:.1f}s")
    print(f"Average time per example: {summary_stats['average_time_per_example']:.1f}s")
    print(f"Average time per step: {summary_stats['average_time_per_iteration']:.1f}s")

print(f"\nDetailed results saved to: {out_csv}")
print(f"Total processing time: {elapsed:.1f}s")

Loading: /Users/yanivgal/dev/ai21/promptsmith/data/documents_train.csv


TextGrad Refining:   0%|          | 0/50 [00:00<?, ?it/s]


[TextGrad] Starting prompt refinement process...

[TextGrad] === Step 1/5 ===
[TextGrad] Running bulletizer with current prompt...
[TextGrad] Current output:
 # Cloud Security Risks and Best Practices

Cloud computing presents various security risks that can impact data integrity and availability.

## Account or Session Hijacking
- **Definition:** Cloud users access data through cloud-based systems.
- **Threats:** User credentials and sessions can be hijacked by attackers.
- **Consequences:** Unauthorized access can lead to data theft, alteration, deletion, or sale.
- **Impact:** Company integrity and prestige can be compromised; legal implications for industries like healthcare.
- **Prevention:** Keeping credentials safe, using two-factor authentication, and monitoring operations can mitigate risks.

---

## Multi-Tenancy
- **Definition:** Multiple users share the same computational resources in cloud computing.
- **Data Security:** Users' data is kept separate despite sharing resour

TextGrad Refining:   2%|▏         | 1/50 [03:50<3:08:26, 230.75s/it]


Feedback summary:
- Fix the one-line summary to follow the required format; it should be a single plain sentence without any additional context.
- Remove redundancy in the presentation of concepts, particularly in the sections on "Account or Session Hijacking" and "Backup," where similar ideas about the importance of security measures are repeated.
- Consolidate statements that convey similar sentiments about the necessity of security measures to enhance clarity and avoid repetition.
=== End Evaluation ===

[TextGrad] Final combined score: 0.88

✅ Completed example 1: Final score = 0.880

[TextGrad] Starting prompt refinement process...

[TextGrad] === Step 1/5 ===
[TextGrad] Running bulletizer with current prompt...
[TextGrad] Current output:
 # Elisa 5G Network Launch in Finland

Elisa has launched its 5G network in 23 locations across Finland.

## 5G Network Availability
- **Locations:** 23 cities including Helsinki, Espoo, Tampere, Vantaa, Oulu, Turku, Jyvaskyla, Lahti, Kuopio, Po

...


Feedback summary:
- Remove redundancy in the sections discussing urban forestry's role in climate action plans. Combine similar points about enhancing community resilience and mitigating climate change effects into a single statement to avoid repetition.
- Include specific details and data points that were omitted, such as the estimated carbon storage in urban soils and the effectiveness of certain tree species, to enhance the depth of coverage.
- Ensure that all key points from the original text are captured, particularly those that provide specific metrics or examples related to urban forestry's impact.
=== End Evaluation ===

[TextGrad] Final combined score: 0.925

✅ Completed example 48: Final score = 0.925

[TextGrad] Starting prompt refinement process...

[TextGrad] === Step 1/5 ===
[TextGrad] Running bulletizer with current prompt...
[TextGrad] Current output:
 # Trends in Past-Year Depression and Help Seeking (2015-2019)

The study analyzes trends in past-year depression and h

TextGrad Refining:  98%|█████████▊| 49/50 [3:55:24<05:08, 308.27s/it]


Feedback summary:
- Fix the omission of the time frame (2015 to 2019) in the help-seeking behavior section, as it is a key detail.
- Improve the output by including more specific details about the results or findings from the analyses, which are currently implied but not explicitly stated.
- Remove redundancy in the structure of the analysis methods, particularly the repeated mention of unadjusted and adjusted analyses across sections. Consolidate this information into a single explanation applicable to all sections.
- Simplify the closing phrases summarizing the findings at the end of each section to avoid repetitive structure.
=== End Evaluation ===

[TextGrad] Final combined score: 0.925

✅ Completed example 49: Final score = 0.925

[TextGrad] Starting prompt refinement process...

[TextGrad] === Step 1/5 ===
[TextGrad] Running bulletizer with current prompt...
[TextGrad] Current output:
 # Symbolic Annihilation and Gender Representation

The under-representation of women in media 

TextGrad Refining: 100%|██████████| 50/50 [3:59:31<00:00, 287.43s/it]


Feedback summary:
- Remove redundancy by consolidating the discussions on the under-representation and trivialization of women, particularly in the "Symbolic Annihilation" and "Coverage of Women's Sports" sections, as both mention similar themes.
- Streamline the overarching theme of limited representation to enhance clarity and reduce verbosity, avoiding reiteration across multiple sections.
- Ensure that each section presents unique points without overlapping content to improve the overall conciseness of the output.
=== End Evaluation ===

[TextGrad] Final combined score: 0.955

✅ Completed example 50: Final score = 0.955

Saving detailed results to /Users/yanivgal/dev/ai21/promptsmith/r_50_5_textgrad_20250924_065818.csv...
Saved 250 rows to /Users/yanivgal/dev/ai21/promptsmith/r_50_5_textgrad_20250924_065818.csv

TEXTGRAD PROCESS COMPLETE
Successfully processed: 50 examples
Average final score: 0.931
Average steps per example: 5.0
Total time: 14371.2s
Average time per example: 287.